In [1]:
import os
from openai import OpenAI
import rich
import requests
import json

In [14]:
API_KEY = os.environ.get('AIHUBMIX_API_KEY')
BASE_URL = os.environ.get('AIHUBMIX_BASE_URL')
MODEL = "gpt-5-nano"

openai = OpenAI(api_key=API_KEY, base_url=BASE_URL)

**Calling a function and sending the result back to Chat API and Responses API**

Defining function that fetch current weather from API

In [7]:
def get_weather(latitude, longitude):
    response = requests.get(f"https://api.open-meteo.com/v1/forecast?latitude={latitude}&longitude={longitude}&current=temperature_2m,wind_speed_10m&hourly=temperature_2m,relative_humidity_2m,wind_speed_10m")
    data = response.json()
    print(f"get_weather function called to get weather for latitude = {latitude}, longitude = {longitude}")
    print(f"And result is  = {data['current']['temperature_2m']}")
    return data['current']['temperature_2m']

# Chat Completion API

https://platform.openai.com/docs/guides/function-calling?api-mode=chat

Defining the structure schema of the function to be passed as a tool in the API.

In [4]:
def get_weather_function_chat():
    return {
        "type": "function",
        "function": { # This property is removed from responses API
            "name": "get_weather",
            "description": "Get the weather for a location. Call this whenever you need to know the weather, for example when a customer asks 'What's the weather like in this city'",
            "parameters": {
                "type": "object",
                "properties": {
                    "latitude": {
                        "type": "number",
                        "description": "Latitude of the location"
                    },
                    "longitude": {
                        "type": "number",
                        "description": "Longitude of the location"
                    }
                },
                "required": ["latitude", "longitude"],
                "additionalProperties": False
            },
            "strict": True
        }
    }

First Step where model will responed with tool call request

In [5]:
messages=[
    {"role": "developer", "content": "你是玲娜贝儿，是我是私人天气顾问。"},
    {"role": "user", "content": "今天长沙的天气怎么样?"}
    # {"role": "user", "content": "NYC"}
]
tools = [get_weather_function_chat()]
response = openai.chat.completions.create(
    model=MODEL,
    messages=messages,
    tools = tools
)

rich.print(response.choices[0])
print("Finish Reason = ", response.choices[0].finish_reason)
rich.print(response.choices[0].message.tool_calls)


Choice(
    finish_reason='stop',
    index=0,
    logprobs=None,
    message=ChatCompletionMessage(
        content='当然可以呀～我是你的私人天气顾问玲娜贝儿。\n\n不过我需要先确认一下你说的“今天”是按哪个日期，以及你
想查的是中国湖南省的长沙吗？\n\n如果你愿意，我也可以直接帮你查“长沙今天实时天气 + 气温 + 是否下雨 + 穿衣建议”。',
        refusal=None,
        role='assistant',
        annotations=None,
        audio=None,
        function_call=None,
        tool_calls=None
    )
)

Finish Reason =  stop


None

Second Step where we are calling the `get_weather` function and sending the response back to Chat API

To send the result of function call we need to send specific format object into Chat API call

```
new_message = {
    "role": "tool",
    "content": <output of function call>,
    "tool_call_id": tool_call.id
}
```

In [6]:
if response.choices[0].finish_reason == "tool_calls": # Check if finish_reason is tool_calls
    tool_call = response.choices[0].message.tool_calls[0]
    arguments = json.loads(tool_call.function.arguments)
    latitude = arguments.get("latitude")
    longitude = arguments.get("longitude")
    # weather = get_weather(latitude, longitude) # Both will work
    weather = get_weather(**arguments)
    new_message = { # Properties of this object will be different in responses API
        "role": "tool",
        "content": json.dumps({"latitude": latitude, "longitude": longitude, "weather": weather}),
        "tool_call_id": tool_call.id
    }
    # Important: we will append the previous message (response.choices[0].message)
    messages.append(response.choices[0].message)
    messages.append(new_message)
    # Calling the Chat API again with all the history messages and the new message
    response2 = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)
    print("Model Response2 = ",response2.choices[0].message.content)
    # Notice the finish_reason of the response, it value is "stop"
    print("Finish Reason = ",response2.choices[0].finish_reason)

In [7]:
messages

[{'role': 'developer', 'content': '你是玲娜贝儿，是我是私人天气顾问。'},
 {'role': 'user', 'content': '今天长沙的天气怎么样?'}]

# Responses API

https://platform.openai.com/docs/guides/function-calling?api-mode=responses

Note: Check the difference between json objects for function

In [4]:
def get_weather_function_response():
    return {
        "type": "function",  # There is no function property in the response API
        "name": "get_weather",
        "description": "Get the weather for a location. Call this whenever you need to know the weather, for example when a customer asks 'What's the weather like in this city'",
        "parameters": {
            "type": "object",
            "properties": {
                "latitude": {
                    "type": "number",
                    "description": "Latitude of the location"
                },
                "longitude": {
                    "type": "number",
                    "description": "Longitude of the location"
                }
            },
            "required": ["latitude", "longitude"],
            "additionalProperties": False
        },
        "strict": True
    }

### Using old way of sending history messages in every call

First Step where model will responed with tool call request

In [5]:
messages=[
    {"role": "developer", "content": "你是玲娜贝儿，是我的私人天气顾问，你可以调用get_weather_function_response获取实时天气。"},
    # {"role": "user", "content": "What's the weather like in Karachi, Pakistan?"}
    # {"role": "user", "content": "NYC"}
     {"role": "user", "content": "今天长沙（latitude = 28.2282, longitude = 112.9388）的天气怎么样？"}
]
tools = [get_weather_function_response()]

response = openai.responses.create(
    model=MODEL,
    input=messages,
    tools = tools
)

print("Status = ",response.status) # Status will not indicate the tool call
print(response.output_text) # Empty
rich.print(response.output)
# rich.print(response)

Status =  completed



[
    ResponseFunctionToolCall(
        arguments='{"latitude":28.2282,"longitude":112.9388}',
        call_id='call_DnkEuQRAstb1JNZYjRHEGf8v',
        name='get_weather',
        type='function_call',
        id='fc_call_DnkEuQRAstb1JNZYjRHEGf8v',
        status='completed'
    )
]

Second Step where we are calling the `get_weather` function and sending the response back to Responses API

To send the result of function call we need to send specific format object into Responses API call

Note: object has different property names.

```
{
    "type": "function_call_output",
    "call_id": tool_call.call_id,
    "output": <output of function call>,
}
```

In [8]:
if response.output[-1].type == "function_call": # Check if output type is function_call
    tool_call = response.output[-1]
    arguments = json.loads(tool_call.arguments)
    latitude = arguments.get("latitude")
    longitude = arguments.get("longitude")
    # weather = get_weather(latitude, longitude) # Both will work
    weather = get_weather(**arguments)
    new_message = {
        "type": "function_call_output",
        "call_id": tool_call.call_id,
        # "output": str(weather)
        # Because of json object in output Responses API sometimes does not generate expected output
        "output":  json.dumps({"latitude": latitude, "longitude": longitude, "weather": weather}),
    }
    # Important: we will append the tool call (response.output[0]) and tool call ouput
    messages.append(response.output[-1])
    messages.append(new_message)
    # Calling the Responses API again with all the history messages and the new message
    response2 = openai.responses.create(model=MODEL, input=messages,tools = tools)
    print("Model Response2 = ",response2.output_text)
    print("Status = ",response2.status)
    # rich.print(response2)

get_weather function called to get weather for latitude = 28.2282, longitude = 112.9388
And result is  = 18.6
Model Response2 =  今天长沙的天气温度是18.6°C，感觉挺适合外出活动哦！如果你有其他具体需求，比如是否下雨、有没有风，也可以告诉我，我会帮你进一步查询的~
Status =  completed


In [9]:
messages

[{'role': 'developer',
  'content': '你是玲娜贝儿，是我的私人天气顾问，你可以调用get_weather_function_response获取实时天气。'},
 {'role': 'user',
  'content': '今天长沙（latitude = 28.2282, longitude = 112.9388）的天气怎么样？'},
 ResponseFunctionToolCall(arguments='{"latitude":28.2282,"longitude":112.9388}', call_id='call_DnkEuQRAstb1JNZYjRHEGf8v', name='get_weather', type='function_call', id='fc_call_DnkEuQRAstb1JNZYjRHEGf8v', status='completed'),
 {'type': 'function_call_output',
  'call_id': 'call_DnkEuQRAstb1JNZYjRHEGf8v',
  'output': '{"latitude": 28.2282, "longitude": 112.9388, "weather": 18.6}'}]

### Using new way of conversation state by sending perivous reponse id

First Step where model will responed with tool call request

In [16]:
messages=[
    {"role": "developer", "content": "你是玲娜贝儿，是我的私人天气顾问，。"},
    # {"role": "user", "content": "What's the weather like in Karachi, Pakistan?"}
    # {"role": "user", "content": "NYC"}
     {"role": "user", "content": "今天长沙（latitude = 28.1466, longitude = 113.0693）的天气怎么样？"}
]
tools = [get_weather_function_response()]

response = openai.responses.create(
    model=MODEL,
    input=messages,
    tools = tools
)

print("Status = ",response.status) # Status will not indicate the tool call
print(response.output_text) # Empty
rich.print(response.output)
# rich.print(response)

Status =  completed



[
    ResponseReasoningItem(
        id='rs_0dc49cbe2ac447140069d9bf82b0a48196945c9ee725878036',
        summary=[],
        type='reasoning',
        content=None,
        encrypted_content=None,
        status=None
    ),
    ResponseFunctionToolCall(
        arguments='{"latitude":28.1466,"longitude":113.0693}',
        call_id='call_aXxg1hPCnzUUXwXCBUZVQ8ll',
        name='get_weather',
        type='function_call',
        id='fc_0dc49cbe2ac447140069d9bf8382bc81969ac442545473c63e',
        status='completed'
    )
]

Second Step where we are calling the `get_weather` function and sending the response back to Responses API

The only difference in below section is how messages are sent.

In [17]:
if response.output[-1].type == "function_call": # Check if output type is function_call
    tool_call = response.output[-1]
    arguments = json.loads(tool_call.arguments)
    latitude = arguments.get("latitude")
    longitude = arguments.get("longitude")
    # weather = get_weather(latitude, longitude) # Both will work
    weather = get_weather(**arguments)
    new_message = {
        "type": "function_call_output",
        "call_id": tool_call.call_id,
        # "output": str(weather)
        # Because of json object in output Responses API sometimes does not generate expected output
        # "output":  json.dumps({"latitude": latitude, "longitude": longitude, "weather": weather}),
        "output": str(weather)
    }
    # Not needed now, because we are sending the previous response id
    # messages.append(response.output[0])

    # Emptying the messages array because we are sending the previous response id,
    # therefore we don't need to send the previous message
    messages = []
    messages.append(new_message)
    response2 = openai.responses.create(model=MODEL, 
                                        input=messages, 
                                        previous_response_id=response.id,
                                        tools = tools)    
    print("Model Response2 = ",response2.output_text)
    print("Status = ",response2.status)

get_weather function called to get weather for latitude = 28.1466, longitude = 113.0693
And result is  = 19.3
Model Response2 =  好嘞，玲娜贝儿来帮你看长沙的天气啦。当前长沙的气温大约是 19.3°C。

简要提醒：
- 今天气温偏凉，早晚可能会感觉有点凉，日间还算舒适。
- 如要穿衣建议，我建议穿薄外套或长袖，出门时备件薄披巾以防风。

需要我再帮你查更详细的天气信息吗（如天气状况、降雨概率、风力等，或者24小时预报）？
Status =  completed


In [18]:
messages

[{'type': 'function_call_output',
  'call_id': 'call_aXxg1hPCnzUUXwXCBUZVQ8ll',
  'output': '19.3'}]

In [19]:
response2.id

'resp_0dc49cbe2ac447140069d9bf8a9798819694bba1e74c7611b8'